In [1]:
%load_ext autoreload
%autoreload 2

# *RTGL* Workflow & Evaluation on *RelBench*

## Contents
1. [Helper Functions](#helper-functions)
2. [F1 Dataset](#f1-dataset)  
    - 2.1 [Entity Classification Tasks](#f1-clas-tasks)
        - 2.1.1 [driver-dnf](#driver-dnf)  
        - 2.1.2 [driver-top3](#driver-top3)  
    - 2.2 [Entity Regression Tasks](#f1-reg-tasks)  
        - 2.2.1 [driver-position](#driver-position)  
3. [Stack-Exchange Q&A Website Dataset](#stack-exchange-dataset)
    - 3.1 [Entity Classification Tasks](#stack-clas-tasks)  
        - 3.1.1 [user-engagement](#user-engagement)  
        - 3.1.2 [user-badge](#user-badge)  
    - 3.2 [Entity Regression Tasks](#stack-reg-tasks)  
        - 3.2.1 [post-votes](#post-votes)  
    - 3.3 [Link Prediction Tasks](#stack-link-tasks)  
        - 3.3.1 [user-post-comment](#user-post-comment)  
        - 3.3.2 [post-post-related](#post-post-related)  

## Overview
This notebook demonstrates the simplicity of using *RTGL* for **Task Generation** in **Relational Deep Learning**.  

The *RelBench* framework is used as the primary source of data and tasks, leveraging its collection of pre-defined tasks to evaluate *RTGL*'s capabilities. 

## 1. Helper Functions <a id="helper-functions"></a>

In this section, we define utility functions to streamline our evaluation workflow.

In [2]:
import pandas as pd
import numpy as np

from relbench.base import BaseTask, Dataset

from rtgl.converter import TConverter

First, we need to generate the same timestamps as in *RelBench*.

In [3]:
def get_timestamps(dataset: Dataset, 
                   timedelta: pd.Timedelta, 
                   num_eval_timestamps: int, 
                   split: str) -> "pd.Series[pd.Timestamp]":
    db = dataset.get_db(upto_test_timestamp=(split != "test"))

    if split == "train":
        start = dataset.val_timestamp - timedelta
        end = db.min_timestamp
        freq = -timedelta
    elif split == "val":
        start = dataset.val_timestamp
        end = min(
            dataset.val_timestamp
            + timedelta * (num_eval_timestamps - 1),
            dataset.test_timestamp - timedelta,
            )
        freq = timedelta
    elif split == "test":
        start = dataset.test_timestamp
        end = min(
            dataset.test_timestamp
            + timedelta * (num_eval_timestamps - 1),
            db.max_timestamp - timedelta,
            )
        freq = timedelta
    else:
        pass

    timestamps = pd.date_range(start=start, end=end, freq=freq)
    return timestamps

Second, we need to pre-process the *pd.DataFrame* obtained from *RelBench* tasks.

Since tasks generated with *RTGL* have pre-defined column names - `fk`, `timestamp`, `label` - we must rename the original *RelBench* columns to ensure the same structure.

In [4]:
def process_df_rb(df_rb: pd.DataFrame,
                  fk: str,
                  timestamp: str,
                  label: str) -> pd.DataFrame:
    renamed_df_rb = df_rb.rename(columns={fk: 'fk',
                                          timestamp: 'timestamp',
                                          label: 'label'})
    df_rb = renamed_df_rb.sort_values(by=['timestamp', 'fk'])

    df_rb['timestamp'] = df_rb['timestamp']

    return df_rb

Third, we need to merge *RTGL* and *RelBench* DataFrames.

This step ensures that the *RTGL* output matches *RelBench*.

In [5]:
def merge_dataframes(df_rb: pd.DataFrame,
                     df_rtgl: pd.DataFrame) -> None:
    # normalization if LIST_DISTINCT was used in the query
    def normalize(x):
        if isinstance(x, (list, np.ndarray, tuple)):
            return tuple(sorted(x))
        elif isinstance(x, (float, np.floating)):
            return round(float(x), 4)
        
        return x

    df_rb['label'] = df_rb['label'].apply(normalize)
    df_rtgl['label'] = df_rtgl['label'].apply(normalize)

    merged = pd.merge(
        df_rb,
        df_rtgl,
        on=['fk', 'timestamp', 'label'],
        how='outer',
        suffixes=('_rb', '_rtgl'),
        indicator=True
    )

    print(f"Only in RelBench:\n {merged[merged['_merge'] == 'left_only']}")
    print(f"Only in RTGL:\n {merged[merged['_merge'] == 'right_only']}")
    print(f"In both:\n {merged[merged['_merge'] == 'both']}")

Last, we need to wrap all the previous steps into a single function.

In [6]:
def check_correctness(dataset: Dataset,
                      task: BaseTask,
                      split: str,
                      rtgl_query: str,
                      fk_col_name: str,
                      timestamp_col_name: str,
                      label_col_name: str) -> None:
    timestamps = get_timestamps(dataset, task.timedelta, task.num_eval_timestamps, split)

    print(f"TIMEDELTA: {task.timedelta}")
    print(f"NUM_EVAL_TIMESTAMPS: {task.num_eval_timestamps}")

    converter = TConverter(dataset.get_db(upto_test_timestamp=(split != "test")), timestamps)
    table_rb = task.get_table(split, mask_input_cols=False)
    df_rb = process_df_rb(table_rb.df, fk_col_name, timestamp_col_name, label_col_name)
    table_rtgl = converter.convert(rtgl_query, execute=True)
    df_rtgl = table_rtgl.df

    print(f"------------------- START {split.upper()} -------------------")
    print(f"RelBench fkeys: {table_rb.fkey_col_to_pkey_table}")
    print(f"RelBench pkey: {table_rb.pkey_col}")
    print(f"RelBench time col: {table_rb.time_col}")
    print(f"RTGL fkeys: {table_rtgl.fkey_col_to_pkey_table}")
    print(f"RTGL pkey: {table_rtgl.pkey_col}")
    print(f"RTGL time col: {table_rtgl.time_col}")
    merge_dataframes(df_rb, df_rtgl)
    print(f"------------------- END {split.upper()} ---------------------")

## 2. F1 Dataset <a id="f1-dataset"></a>

In this section, we attempt to generate the same tasks from the `F1 Dataset` which are already pre-defined in *RelBench*.

In [7]:
from relbench.datasets import get_dataset
from relbench.tasks import get_task, get_task_names

In [8]:
dataset_f1 = get_dataset(name="rel-f1", download=True)
get_task_names("rel-f1")

['driver-position',
 'driver-dnf',
 'driver-top3',
 'driver-circuit-compete',
 'results-position',
 'qualifying-position']

### 2.1 Entity Classification Tasks <a id="f1-clas-tasks"></a>

#### 2.1.1 driver-dnf <a id="driver-dnf"></a>

Task Description: For each driver predict the if they will DNF (did not finish) a race in the next 1 month.

Note: There is a mistake in the *RelBench* task generation logic for this specific task. (See the proof below).

In [9]:
task_f1_dnf = get_task("rel-f1", "driver-dnf", download=False)

In [10]:
rtgl_query = """
    PREDICT MAX(results.statusId, 0, 30, DAYS) != 1
    FOR EACH drivers.*
    WHERE COUNT(results.*, -365, 0, DAYS) != 0
    ASSUMING MAX(results.statusID, 0, 30, DAYS) IS NOT NULL 
    ;
"""

In [12]:
# TRAIN

check_correctness(dataset=dataset_f1,
                  task=task_f1_dnf,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="did_not_finish")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
SQL query executed in 0.49 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
        timestamp   fk label     _merge
0     2000-02-27    1     1  left_only
42    2001-02-21    3     1  left_only
50    2003-02-11    3     0  left_only
68    2001-02-21    7     1  left_only
102   2004-06-05    9     1  left_only
...          ...  ...   ...        ...
11406 1950-08-18  802     0  left_only
11407 1950-05-20  803     1  left_only
11408 1953-05-04  804     1  left_only
11409 1954-05-29  805     1  left_only
11410 1956-01-19  806     1  left_only

[1022 rows x 4 columns]
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
        timestamp   fk label _merge
1     2000-03-28    1     1   both
2     2000-04-27    1     1   bo

In [13]:
# VAL

check_correctness(dataset=dataset_f1,
                  task=task_f1_dnf,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="did_not_finish")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
SQL query executed in 0.07 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
      timestamp  fk label     _merge
0   2007-02-20   0     0  left_only
34  2006-02-25   2     1  left_only
78  2007-02-20   4     1  left_only
88  2007-10-18   5     1  left_only
90  2008-03-16   6     1  left_only
117 2006-07-25   8     1  left_only
130 2008-03-16   9     1  left_only
157 2008-03-16  11     1  left_only
236 2007-02-20  15     1  left_only
298 2005-03-02  18     1  left_only
299 2007-02-20  18     1  left_only
309 2007-05-21  19     0  left_only
392 2005-04-01  23     0  left_only
411 2005-04-01  24     0  left_only
412 2007-02-20  24     1  left_only
420 2006-02-25  25     1  left_only
434 2005-03-02  26     1  left_only
455 2007-07-20  27     1  left_only
456 

In [14]:
# TEST

check_correctness(dataset=dataset_f1,
                  task=task_f1_dnf,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="did_not_finish")

Loading Database object from /home/kolesole/.cache/relbench/rel-f1/db...
Done in 0.05 seconds.
TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
SQL query executed in 0.08 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
      timestamp   fk label     _merge
123 2012-02-20    7     0  left_only
236 2013-03-16   15     1  left_only
360 2010-06-30   28     1  left_only
364 2010-03-02   29     0  left_only
392 2010-08-29   31     1  left_only
394 2010-03-02   36     1  left_only
412 2011-03-27   38     1  left_only
463 2012-02-20  153     1  left_only
502 2010-03-02  807     1  left_only
511 2012-02-20  807     1  left_only
522 2010-03-02  808     1  left_only
550 2010-03-02  809     1  left_only
559 2010-03-02  810     1  left_only
582 2010-03-02  811     1  left_only
588 2011-03-27  812     1  left_only


To demonstrate why the mismatch appears, let's examine the original *RelBench* query for this task:
```sql
SELECT
    t.timestamp as date,
    re.driverId as driverId,
    MAX(CASE WHEN re.statusId != 1 THEN 1 ELSE 0 END) AS did_not_finish
FROM
    timestamp_df t
LEFT JOIN
    results re
ON
    re.date <= t.timestamp + INTERVAL '{self.timedelta}'
    and re.date  > t.timestamp
WHERE
    -- Data Leakage: missing upper bound(<= t.timestamp)
    re.driverId IN (
        SELECT DISTINCT driverId
        FROM results
        WHERE date > t.timestamp - INTERVAL '1 year'
    )
GROUP BY t.timestamp, re.driverId
```

It easily to note that *RelBench* team forgot to add an upper bound for the driver filtering.  

This allows `data leakage` from the future, as the query can see if a driver will participate in a race after the prediction timestamp.

#### 2.1.2 driver-top3 <a id="driver-top3"></a>

Task Description: For each driver predict if they will qualify in the top-3 for a race in the next 1 month. 

In [15]:
task_f1_top3 = get_task("rel-f1", "driver-top3", download=False)

In [16]:
rtgl_query = """
    PREDICT MIN(qualifying.position, 0, 30, DAYS) <= 3
    FOR EACH drivers.*
    ASSUMING MIN(qualifying.position, 0, 30, DAYS) IS NOT NULL;
"""

In [17]:
# TRAIN

check_correctness(dataset=dataset_f1,
                  task=task_f1_top3,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="qualifying")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
SQL query executed in 0.19 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
       timestamp   fk label _merge
0    2000-02-27    1     0   both
1    2000-03-28    1     0   both
2    2000-09-24    1     0   both
3    2001-09-19    1     0   both
4    2002-02-16    1     0   both
...         ...  ...   ...    ...
1348 1994-08-27  112     0   both
1349 1994-08-27  113     0   both
1350 1994-09-26  114     0   both
1351 1994-10-26  114     0   both
1352 1994-10-26  115     0   both

[1353 rows x 4 columns]
------------------- END TRAIN ---------------------


In [18]:
# VAL

check_correctness(dataset=dataset_f1,
                  task=task_f1_top3,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="qualifying")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
SQL query executed in 0.04 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp  fk label _merge
0   2007-02-20   0     0   both
1   2007-03-22   0     1   both
2   2007-04-21   0     0   both
3   2007-05-21   0     1   both
4   2007-06-20   0     1   both
..         ...  ..   ...    ...
583 2005-05-31  39     0   both
584 2005-06-30  39     0   both
585 2005-05-31  40     0   both
586 2005-08-29  41     0   both
587 2005-09-28  41     0   both

[588 rows x 4 columns]
------------------- END VAL ---------------------


In [19]:
# TEST

check_correctness(dataset=dataset_f1,
                  task=task_f1_top3,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="qualifying")

TIMEDELTA: 30 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
SQL query executed in 0.05 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp   fk label _merge
0   2010-03-02    0     0   both
1   2010-04-01    0     0   both
2   2010-05-01    0     1   both
3   2010-05-31    0     1   both
4   2010-06-30    0     0   both
..         ...  ...   ...    ...
721 2013-03-16  819     0   both
722 2013-03-16  820     0   both
723 2013-03-16  821     0   both
724 2013-03-16  822     0   both
725 2013-03-16  823     0   both

[726 rows x 4 columns]
------------------- END TEST ---------------------


### 2.2 Entity Regression Tasks <a id="f1-reg-tasks"></a>

#### 2.2.1 driver-position <a id="driver-position"></a>

Task Description: Predict the average finishing position of each driver all races in the next 2 months. 

In [20]:
task_f1_pos = get_task("rel-f1", "driver-position", download=False)

In [21]:
rtgl_query = """
    PREDICT AVG(results.positionOrder, 0, 60, DAYS)
    FOR EACH drivers.*;
"""

In [22]:
# TRAIN

check_correctness(dataset=dataset_f1,
                  task=task_f1_pos,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="position")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
SQL query executed in 0.36 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
       timestamp   fk    label _merge
0    2000-01-28    1  14.0000   both
1    2000-03-28    1  17.6667   both
2    2000-05-27    1  13.7500   both
3    2000-07-26    1  15.0000   both
4    2000-09-24    1  19.5000   both
...         ...  ...      ...    ...
7448 1950-06-19  801   6.0000   both
7449 1950-08-18  802   2.0000   both
7450 1953-04-04  804  17.0000   both
7451 1954-05-29  805  30.0000   both
7452 1956-01-19  806   6.0000   both

[7453 rows x 4 columns]
------------------- END TRAIN ---------------------


In [23]:
# VAL

check_correctness(dataset=dataset_f1,
                  task=task_f1_pos,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="position")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
SQL query executed in 0.03 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp   fk    label _merge
0   2007-02-20    0   2.3333   both
1   2007-04-21    0   1.5000   both
2   2007-06-20    0   4.0000   both
3   2007-08-19    0   6.2000   both
4   2007-10-18    0   7.0000   both
..         ...  ...      ...    ...
494 2009-08-08  152  17.4000   both
495 2009-10-07  152  17.0000   both
496 2009-08-08  153  16.8000   both
497 2009-10-07  153  15.5000   both
498 2009-10-07  154   8.0000   both

[499 rows x 4 columns]
------------------- END VAL ---------------------


In [24]:
# TEST

check_correctness(dataset=dataset_f1,
                  task=task_f1_pos,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="driverId",
                  timestamp_col_name="date",
                  label_col_name="position")

TIMEDELTA: 60 days 00:00:00
NUM_EVAL_TIMESTAMPS: 40
SQL query executed in 0.04 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'driverId': 'drivers'}
RelBench pkey: None
RelBench time col: date
RTGL fkeys: {'fk': 'drivers'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp   fk    label _merge
0   2010-03-02    0   4.2500   both
1   2010-05-01    0   4.6000   both
2   2010-06-30    0   8.6667   both
3   2010-08-29    0  10.0000   both
4   2010-10-28    0   3.0000   both
..         ...  ...      ...    ...
755 2016-05-29  835  17.0000   both
756 2016-01-30  836  19.0000   both
757 2016-03-30  836  19.2500   both
758 2016-05-29  836  18.0000   both
759 2016-03-30  837  10.0000   both

[760 rows x 4 columns]
------------------- END TEST ---------------------


## 3. Stack-Exchange Q&A Website Dataset <a id="stack-exchange-dataset"></a>

In this section, we attempt to generate the same tasks from the `Stack-Exchange Q&A Website Dataset` which are already pre-defined in *RelBench*.

In [25]:
from relbench.datasets import get_dataset
from relbench.tasks import get_task, get_task_names

In [26]:
dataset_stack = get_dataset(name="rel-stack", download=True)
get_task_names("rel-stack")

['user-engagement',
 'post-votes',
 'user-badge',
 'user-post-comment',
 'post-post-related',
 'badges-class']

### 3.1 Entity Classification Tasks <a id="stack-clas-tasks"></a>

#### 3.1.1 user-engagement <a id="user-engagement"></a>

Task Description: For each user predict if a user will make any votes, posts, or comments in the next 3 months. 

Note: There is a mistake in the *RelBench* task generation logic for this specific task. (See the proof below).

In [27]:
task_stack_engage = get_task("rel-stack", "user-engagement", download=False)

In [28]:
rtgl_query = """
     PREDICT COUNT(votes.*, 0, 91, DAYS) != 0
          OR COUNT(posts.*, 0, 91, DAYS) != 0
          OR COUNT(comments.*, 0, 91, DAYS) != 0
     FOR EACH users.*
     WHERE COUNT(votes.*, -inf, 0, DAYS) != 0
           OR COUNT(posts.*, -inf, 0, DAYS) != 0
           OR COUNT(comments.*, -inf, 0, DAYS) != 0;
"""

In [29]:
# TRAIN

check_correctness(dataset=dataset_stack,
                  task=task_stack_engage,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="OwnerUserId",
                  timestamp_col_name="timestamp",
                  label_col_name="contribution")

Loading Database object from /home/kolesole/.cache/relbench/rel-stack/db...
Done in 7.57 seconds.
TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 2.03 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'OwnerUserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
          timestamp      fk label     _merge
1249    2010-07-15      37     1  left_only
7790    2010-07-15     270     1  left_only
20439   2010-01-14     855     0  left_only
20440   2010-04-15     855     0  left_only
20441   2010-07-15     855     0  left_only
...            ...     ...   ...        ...
1360845 2020-07-02  251501     0  left_only
1360846 2019-10-03  252540     0  left_only
1360847 2020-01-02  252540     0  left_only
1360848 2020-04-02  252540     0  left_only
1360849 2020-07-02  252540     0  left_only

[1613 rows x 4 columns]
Only in RTGL:
 Empty DataFrame
C

To demonstrate why the mismatch appears, let's examine the user with `id=37`.

In [29]:
users_table_df = dataset_stack.get_db().table_dict['users'].df
users_table_df[users_table_df['Id'] == 37]

Loading Database object from /home/kolesole/.cache/relbench/rel-stack/db...
Done in 7.17 seconds.


,Id,AccountId,DisplayName,Location,ProfileImageUrl,WebsiteUrl,AboutMe,CreationDate
37,37,9084.0,hadley,"Houston, TX",NaN,http://hadley.nz,I'm an assistant professor of Statistics at Ri...,2010-07-19 19:13:09.870


In the resulting table, we can observe a record with `fk=37` and `timestamp=2010-07-15`. However, in the original users table, `CreationDate=2010-07-19` for this user.

This confirms a `data leakage` from the future, as the task includes users who had not yet been created at the time of the prediction.

In [30]:
# VAL

check_correctness(dataset=dataset_stack,
                  task=task_stack_engage,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="OwnerUserId",
                  timestamp_col_name="timestamp",
                  label_col_name="contribution")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.25 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'OwnerUserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
        timestamp      fk label     _merge
85825 2020-10-01  247497     0  left_only
85826 2020-10-01  247505     1  left_only
85827 2020-10-01  248036     0  left_only
85828 2020-10-01  248388     0  left_only
85829 2020-10-01  249024     1  left_only
85830 2020-10-01  249067     0  left_only
85831 2020-10-01  249601     0  left_only
85832 2020-10-01  250052     1  left_only
85833 2020-10-01  250509     1  left_only
85834 2020-10-01  250586     1  left_only
85835 2020-10-01  251501     0  left_only
85836 2020-10-01  252540     0  left_only
85837 2020-10-01  252669     0  left_only
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
       

In [31]:
# TEST

check_correctness(dataset=dataset_stack,
                  task=task_stack_engage,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="OwnerUserId",
                  timestamp_col_name="timestamp",
                  label_col_name="contribution")

Loading Database object from /home/kolesole/.cache/relbench/rel-stack/db...
Done in 7.61 seconds.
TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.27 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'OwnerUserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
        timestamp      fk label _merge
0     2021-01-01       0     1   both
1     2021-01-01       2     0   both
2     2021-01-01       4     0   both
3     2021-01-01       5     0   both
4     2021-01-01       6     0   both
...          ...     ...   ...    ...
88132 2021-01-01  255341     0   both
88133 2021-01-01  255347     0   both
88134 2021-01-01  255351     1   both
88135 2021-01-01  255354     0   both
88136 2021-

#### 3.1.2 user-badge <a id="user-badge"></a>

Task Description: For each user predict if a user will receive a new badge in the next 3 months. 

In [32]:
task_stack_badge = get_task("rel-stack", "user-badge", download=False)

In [33]:
rtgl_query = """
    PREDICT COUNT(badges.*, 0, 91, DAYS) != 0
    FOR EACH users.*;
"""

In [34]:
# TRAIN

check_correctness(dataset=dataset_stack,
                  task=task_stack_badge,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="WillGetBadge")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.80 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'UserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp      fk label _merge
0       2010-10-14       0     0   both
1       2011-01-13       0     0   both
2       2011-04-14       0     0   both
3       2011-07-14       0     0   both
4       2011-10-13       0     0   both
...            ...     ...   ...    ...
3386271 2020-07-02  239940     0   both
3386272 2020-07-02  239941     0   both
3386273 2020-07-02  239942     0   both
3386274 2020-07-02  239943     0   both
3386275 2020-07-02  239944     1   both

[3386276 rows x 4 columns]
------------------- END

In [35]:
# VAL

check_correctness(dataset=dataset_stack,
                  task=task_stack_badge,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="WillGetBadge")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.13 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'UserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2020-10-01       0     0   both
1      2020-10-01       1     0   both
2      2020-10-01       2     0   both
3      2020-10-01       3     0   both
4      2020-10-01       4     0   both
...           ...     ...   ...    ...
247393 2020-10-01  247393     0   both
247394 2020-10-01  247394     1   both
247395 2020-10-01  247395     0   both
247396 2020-10-01  247396     0   both
247397 2020-10-01  247397     1   both

[247398 rows x 4 columns]
------------------- END VAL ----------

In [36]:
# TEST

check_correctness(dataset=dataset_stack,
                  task=task_stack_badge,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="WillGetBadge")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.15 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'UserId': 'users'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2021-01-01       0     0   both
1      2021-01-01       1     0   both
2      2021-01-01       2     0   both
3      2021-01-01       3     0   both
4      2021-01-01       4     1   both
...           ...     ...   ...    ...
255355 2021-01-01  255355     1   both
255356 2021-01-01  255356     0   both
255357 2021-01-01  255357     0   both
255358 2021-01-01  255358     1   both
255359 2021-01-01  255359     1   both

[255360 rows x 4 columns]
------------------- END TEST --------

### 3.2 Entity Regression Tasks <a id="stack-reg-tasks"></a>

#### 3.2.1 post-votes <a id="post-votes"></a>

Task Description: For each user post predict how many votes it will receive in the next 3 months 

In [37]:
task_stack_post_votes = get_task("rel-stack", "post-votes", download=False)

In [38]:
rtgl_query = """
    PREDICT COUNT_DISTINCT(votes.* WHERE votes.votetypeid == 2, 0, 91, DAYS)
    FOR EACH posts.* WHERE posts.PostTypeId == 1
                        AND posts.OwnerUserId IS NOT NULL
                        AND posts.OwnerUserId != -1
    ;
"""

In [39]:
# TRAIN

check_correctness(dataset=dataset_stack,
                  task=task_stack_post_votes,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="popularity")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.91 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp      fk  label _merge
0       2009-04-16       0      0   both
1       2009-07-16       0      0   both
2       2009-10-15       0      0   both
3       2010-01-14       0      0   both
4       2010-04-15       0      0   both
...            ...     ...    ...    ...
2453916 2020-07-02  315139      0   both
2453917 2020-07-02  315140      0   both
2453918 2020-07-02  315142      0   both
2453919 2020-07-02  315144      0   both
2453920 2020-07-02  315145      0   both

[2453921 rows x 4 columns]
-----------

In [40]:
# VAL

check_correctness(dataset=dataset_stack,
                  task=task_stack_post_votes,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="popularity")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.12 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk  label _merge
0      2020-10-01       0      4   both
1      2020-10-01      19      2   both
2      2020-10-01      23      2   both
3      2020-10-01      24      1   both
4      2020-10-01      25      1   both
...           ...     ...    ...    ...
156211 2020-10-01  324972      0   both
156212 2020-10-01  324973      0   both
156213 2020-10-01  324976      0   both
156214 2020-10-01  324977      0   both
156215 2020-10-01  324980      0   both

[156216 rows x 4 columns]
------------------- END VA

In [41]:
# TEST

check_correctness(dataset=dataset_stack,
                  task=task_stack_post_votes,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="popularity")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.12 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk  label _merge
0      2021-01-01       0      2   both
1      2021-01-01      19      0   both
2      2021-01-01      23      0   both
3      2021-01-01      24      0   both
4      2021-01-01      25      0   both
...           ...     ...    ...    ...
160898 2021-01-01  333883      0   both
160899 2021-01-01  333885      0   both
160900 2021-01-01  333886      0   both
160901 2021-01-01  333887      0   both
160902 2021-01-01  333891      0   both

[160903 rows x 4 columns]
------------------- END T

### 3.3 Link Prediction Tasks <a id="stack-link-tasks"></a>

#### 3.3.1 user-post-comment <a id="user-post-comment"></a>

Task Description: Predict a list of existing posts that a user will comment in the next two months. 

Note: There is a mistake in the *RelBench* task generation logic for this specific task. (See the proof below).

In [42]:
task_stack_post_comm = get_task("rel-stack", "user-post-comment", download=False)

In [43]:
rtgl_query = """
    PREDICT LIST_DISTINCT(comments.PostId 
        WHERE posts.owneruserid IS NOT NULL
          AND posts.owneruserid != -1, 0, 91, DAYS)
    FOR EACH users.*;
"""

In [44]:
# TRAIN

check_correctness(dataset=dataset_stack,
                  task=task_stack_post_comm,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="PostId")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 1.03 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'UserId': 'users', 'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
        timestamp      fk                                    label     _merge
1116  2010-10-14    1141  (779, 866, 924, 1081, 1189, 2540, 2730)  left_only
1152  2010-10-14    1172             (321, 821, 1339, 1383, 2080)  left_only
1179  2010-10-14    1187                             (2289, 2851)  left_only
1297  2010-10-14    1324                              (790, 2927)  left_only
1334  2010-10-14    1365                                  (2544,)  left_only
...          ...     ...                                      ...        ...
21234 2020-07-02  246768                                (246493,)  left_only
21235 2020-07-02  246774           

To demonstrate why the mismatch appears, let's examine the user with `id=1141`.

In [45]:
users_table_df = dataset_stack.get_db().table_dict['users'].df
users_table_df[users_table_df['Id'] == 1141]

,Id,AccountId,DisplayName,Location,ProfileImageUrl,WebsiteUrl,AboutMe,CreationDate
1141,1141,224447.0,vqv,"Columbus, OH",NaN,http://vince.vu,NaN,2010-10-22 13:50:47.390


In the resulting table, we can observe a record with `fk=1141` and `timestamp=2010-10-14`. However, in the original users table, `CreationDate=2010-10-22` for this user.

This confirms a `data leakage` from the future, as the task includes users who had not yet been created at the time of the prediction.

In [45]:
# VAL

check_correctness(dataset=dataset_stack,
                  task=task_stack_post_comm,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="PostId")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.19 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'UserId': 'users', 'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
      timestamp      fk         label     _merge
725 2020-10-01  247437     (324215,)  left_only
726 2020-10-01  247530       (3816,)  left_only
727 2020-10-01  247549     (125837,)  left_only
728 2020-10-01  247557     (219891,)  left_only
729 2020-10-01  247577     (236963,)  left_only
..         ...     ...           ...        ...
820 2020-10-01  254979     (131380,)  left_only
821 2020-10-01  255019      (92562,)  left_only
822 2020-10-01  255174  (935, 42103)  left_only
823 2020-10-01  255179     (286021,)  left_only
824 2020-10-01  255211     (310316,)  left_only

[100 rows x 4 columns]
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, f

In [46]:
# TEST

check_correctness(dataset=dataset_stack,
                  task=task_stack_post_comm,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="UserId",
                  timestamp_col_name="timestamp",
                  label_col_name="PostId")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.24 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'UserId': 'users', 'PostId': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'users', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp      fk                                     label _merge
0   2021-01-01      22                          (101761, 331974)   both
1   2021-01-01     166                                  (50110,)   both
2   2021-01-01     211                                 (333776,)   both
3   2021-01-01     221                                 (295476,)   both
4   2021-01-01     287                                   (2733,)   both
..         ...     ...                                       ..

#### 3.3.2 post-post-related <a id="post-post-related"></a>

Task Description: Predict a list of existing posts that users will link a given post to in the next two months.

In [47]:
task_stack_post_post = get_task("rel-stack", "post-post-related", download=False)

In [57]:
rtgl_query = """
    WITH links_posts AS (postLinks.PostId->posts.Id)
    PREDICT LIST_DISTINCT(links_posts.RelatedPostId, 0, 91, DAYS)
    FOR EACH posts.*;
"""

In [58]:
# TRAIN

check_correctness(dataset=dataset_stack,
                  task=task_stack_post_post,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="postLinksIdList")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.45 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'PostId': 'posts', 'postLinksIdList': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
       timestamp      fk             label _merge
0    2012-01-12      28           (1427,)   both
1    2015-04-09      74           (4935,)   both
2    2019-10-03      74           (4935,)   both
3    2011-04-14      88           (1574,)   both
4    2016-01-07      90            (146,)   both
...         ...     ...               ...    ...
5850 2020-07-02  315039  (159209, 168163)   both
5851 2020-07-02  315054         (244965,)   both
5852 2020-07-02  315060          (98531,)   

In [59]:
# VAL

check_correctness(dataset=dataset_stack,
                  task=task_stack_post_post,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="postLinksIdList")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.10 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'PostId': 'posts', 'postLinksIdList': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp      fk                     label _merge
0   2020-10-01     128                 (106097,)   both
1   2020-10-01     995                  (63486,)   both
2   2020-10-01    1574                     (88,)   both
3   2020-10-01    5030                  (20350,)   both
4   2020-10-01    8848                  (49884,)   both
..         ...     ...                       ...    ...
221 2020-10-01  324945                 (122765,)   both
222 2020-10-01  324967                 

In [60]:
# TEST

check_correctness(dataset=dataset_stack,
                  task=task_stack_post_post,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="PostId",
                  timestamp_col_name="timestamp",
                  label_col_name="postLinksIdList")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.08 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'PostId': 'posts', 'postLinksIdList': 'posts'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'posts', 'label': 'posts'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
      timestamp      fk                     label _merge
0   2021-01-01      43           (27860, 243921)   both
1   2021-01-01      51            (4010, 146052)   both
2   2021-01-01     210                 (328571,)   both
3   2021-01-01     342                   (7282,)   both
4   2021-01-01    1431                 (297150,)   both
..         ...     ...                       ...    ...
253 2021-01-01  333611          (114400, 189819)   both
254 2021-01-01  333637                

## 4. Amazon e-commerce database <a id="amazon-dataset"></a>

In this section, we attempt to generate the same tasks from the `Amazon e-commerce database` which are already pre-defined in *RelBench*.

In [61]:
from relbench.datasets import get_dataset
from relbench.tasks import get_task, get_task_names

In [62]:
dataset_amazon = get_dataset(name="rel-amazon", download=True)
get_task_names("rel-amazon")

['user-churn',
 'user-ltv',
 'item-churn',
 'item-ltv',
 'user-item-purchase',
 'user-item-rate',
 'user-item-review',
 'review-rating']

### 4.1 Entity Classification Tasks

#### 4.1.1 user-churn <a id="user-engagement"></a>

Task Description: For each user, predict 1 if the customer does not review any product in the next 3 months, and 0 otherwise.  

In [63]:
task_amazon_user_churn = get_task("rel-amazon", "user-churn", download=False)

In [64]:
rtgl_query = """
     PREDICT COUNT(review.*, 0, 91, DAYS) == 0
     FOR EACH customer.*
     WHERE COUNT(review.*, -91, 0, DAYS) != 0;
"""

In [65]:
# TRAIN

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_churn,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="churn")

Loading Database object from /home/kolesole/.cache/relbench/rel-amazon/db...
Done in 38.78 seconds.
TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 14.94 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp       fk label _merge
0       2008-07-10        0     1   both
1       2011-01-06        0     1   both
2       2011-07-07        0     1   both
3       2012-04-05        0     0   both
4       2012-07-05        0     0   both
...            ...      ...   ...    ...
4708378 2015-01-01  1850157     1   both
4708379 2015-01-01  1850158     1   both
4708380 2015-04-02  1850161     1   both
4708381 2014-10-02  1850171     1   both
4708382 2013-04-04  1850183     1   both

[4708383 rows x 4 columns]
------------------- END TRAIN --------------------

In [66]:
# VAL

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_churn,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="churn")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.56 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk label _merge
0      2015-10-01        3     0   both
1      2015-10-01        5     0   both
2      2015-10-01       19     1   both
3      2015-10-01       20     1   both
4      2015-10-01       21     1   both
...           ...      ...   ...    ...
409787 2015-10-01  1850080     1   both
409788 2015-10-01  1850086     1   both
409789 2015-10-01  1850102     1   both
409790 2015-10-01  1850104     1   both
409791 2015-10-01  1850149     1   both

[409792 rows x 4 columns]
---------------

In [67]:
# TEST

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_churn,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="churn")

Loading Database object from /home/kolesole/.cache/relbench/rel-amazon/db...
Done in 38.19 seconds.
TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.56 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk label _merge
0      2016-01-01        2     0   both
1      2016-01-01        3     1   both
2      2016-01-01        5     0   both
3      2016-01-01       17     1   both
4      2016-01-01       23     0   both
...           ...      ...   ...    ...
351880 2016-01-01  1850119     1   both
351881 2016-01-01  1850120     1   both
351882 2016-01-01  1850121     1   both
351883 2016-01-01  18

#### 4.1.2 item-churn <a id="user-engagement"></a>

Task Description:  For each product, predict 1 if the product does not receive any reviews in the next 3 months. 

In [68]:
task_amazon_item_churn = get_task("rel-amazon", "item-churn", download=False)

In [69]:
rtgl_query = """
     PREDICT COUNT (review.*, 0, 91, DAYS) == 0
     FOR EACH product.*
     WHERE COUNT(review.*, -91, 0, DAYS) != 0;
"""

In [70]:
# TRAIN

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_item_churn,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="product_id",
                  timestamp_col_name="timestamp",
                  label_col_name="churn")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 8.60 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp      fk label _merge
0       2013-01-03       0     0   both
1       2013-04-04       0     0   both
2       2013-07-04       0     0   both
3       2013-10-03       0     0   both
4       2014-01-02       0     0   both
...            ...     ...   ...    ...
2536009 2014-01-02  506009     1   both
2536010 2014-10-02  506009     0   both
2536011 2015-01-01  506009     1   both
2536012 2009-01-08  506010     1   both
2536013 2013-01-03  506010     1   both

[2536014 rows x 4 columns]
------------------- END TRAIN ---------------------


In [71]:
# VAL

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_item_churn,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="product_id",
                  timestamp_col_name="timestamp",
                  label_col_name="churn")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.31 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2015-10-01       0     0   both
1      2015-10-01       1     1   both
2      2015-10-01       2     1   both
3      2015-10-01       4     0   both
4      2015-10-01       5     0   both
...           ...     ...   ...    ...
177684 2015-10-01  505996     0   both
177685 2015-10-01  505998     0   both
177686 2015-10-01  505999     0   both
177687 2015-10-01  506000     0   both
177688 2015-10-01  506002     0   both

[177689 rows x 4 columns]
------------------- END VAL --

In [72]:
# TEST

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_item_churn,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="product_id",
                  timestamp_col_name="timestamp",
                  label_col_name="churn")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.35 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk label _merge
0      2016-01-01       0     0   both
1      2016-01-01       3     1   both
2      2016-01-01       4     0   both
3      2016-01-01       5     1   both
4      2016-01-01       6     0   both
...           ...     ...   ...    ...
166837 2016-01-01  505996     0   both
166838 2016-01-01  505998     0   both
166839 2016-01-01  505999     0   both
166840 2016-01-01  506000     0   both
166841 2016-01-01  506002     0   both

[166842 rows x 4 columns]
------------------- END TEST 

### 4.2 Entity Regression Tasks <a id="amazon-reg-tasks"></a>

#### 4.2.1 user-ltv <a id="user-engagement"></a>

Task Description: For each user, predict the $ value of the total number of products they buy and review in the next 3 months. 

In [73]:
task_amazon_user_ltv = get_task("rel-amazon", "user-ltv", download=False)

In [74]:
rtgl_query = """
     PREDICT SUM(product.price, 0, 91, DAYS)
     FOR EACH customer.*
     WHERE COUNT(review.*, -91, 0, DAYS) != 0;
"""

In [75]:
# TRAIN

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_ltv,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="ltv")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 14.52 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp       fk  label _merge
0       2008-07-10        0   0.00   both
1       2011-01-06        0   0.00   both
2       2011-07-07        0   0.00   both
3       2012-04-05        0   6.64   both
4       2012-07-05        0  23.99   both
...            ...      ...    ...    ...
4708378 2015-01-01  1850157   0.00   both
4708379 2015-01-01  1850158   0.00   both
4708380 2015-04-02  1850161   0.00   both
4708381 2014-10-02  1850171   0.00   both
4708382 2013-04-04  1850183   0.00   both

[4708383 rows x 4 columns]
------------------- END TRAIN --------

In [76]:
# VAL

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_ltv,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="ltv")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.74 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk  label _merge
0      2015-10-01        3  27.45   both
1      2015-10-01        5  28.20   both
2      2015-10-01       19   0.00   both
3      2015-10-01       20   0.00   both
4      2015-10-01       21   0.00   both
...           ...      ...    ...    ...
409787 2015-10-01  1850080   0.00   both
409788 2015-10-01  1850086   0.00   both
409789 2015-10-01  1850102   0.00   both
409790 2015-10-01  1850104   0.00   both
409791 2015-10-01  1850149   0.00   both

[409792 rows x 4 columns]
---

In [77]:
# TEST

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_ltv,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="ltv")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.85 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'customer_id': 'customer'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk   label _merge
0      2016-01-01        2   20.32   both
1      2016-01-01        3    0.00   both
2      2016-01-01        5  139.64   both
3      2016-01-01       17    0.00   both
4      2016-01-01       23   49.09   both
...           ...      ...     ...    ...
351880 2016-01-01  1850119    0.00   both
351881 2016-01-01  1850120    0.00   both
351882 2016-01-01  1850121    0.00   both
351883 2016-01-01  1850122    0.00   both
351884 2016-01-01  1850134    0.00   both

[351885 rows x 4

#### 4.2.2 item-ltv <a id="user-engagement"></a>

Task Description: For each product, predict the $ value of the total number purchases and reviews it recieves in the next 3 months.

In [101]:
task_amazon_item_ltv = get_task("rel-amazon", "item-ltv", download=False)

In [104]:
rtgl_query = """
     WITH product_product AS (product.product_id->review.product_id->product.product_id)
     PREDICT SUM(product_product.price, 0, 91, DAYS)
     FOR EACH product.product_id 
     ASSUMING COUNT(review.*, 0, 91, DAYS) != 0;
"""

In [102]:
rtgl_query = """
     WITH product_product AS (product.product_id->review.product_id->product.product_id)
     PREDICT SUM(product_product.price, 0, 91, DAYS)
     FOR EACH product.product_id 
     ASSUMING SUM(product_product.price, 0, 91, DAYS) == 100;
"""

In [ ]:
rtgl_query = """
     PREDICT SUM(
     [
        SELECT
            p.product_id,
            p.price,
            r.review_time
        FROM
            product p, 
            review r
        WHERE 
            p.product_id = r.product_id
     ]{new_products}
      {}
      {product_id->product}
      {}
      {review_time}.price, 0, 91, DAYS)
     FOR EACH product.product_id
     ASSUMING COUNT(review.*, 0, 91, DAYS) != 0
     ;
"""

In [105]:
# TRAIN

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_item_ltv,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="product_id",
                  timestamp_col_name="timestamp",
                  label_col_name="ltv")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 8.91 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp      fk    label _merge
0       2012-10-04       0   899.99   both
1       2013-01-03       0  2197.65   both
2       2013-04-04       0  1737.19   both
3       2013-07-04       0  2030.21   both
4       2013-10-03       0  1862.77   both
...            ...     ...      ...    ...
2707674 2013-10-03  506009    85.86   both
2707675 2014-07-03  506009   171.72   both
2707676 2014-10-02  506009   171.72   both
2707677 2008-10-09  506010    17.87   both
2707678 2012-10-04  506010    17.87   both

[2707679 rows x 4 columns]
------------------- END TRAIN 

In [82]:
# VAL

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_item_ltv,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="product_id",
                  timestamp_col_name="timestamp",
                  label_col_name="ltv")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.32 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk    label _merge
0      2015-10-01       0  2344.16   both
1      2015-10-01       3   159.95   both
2      2015-10-01       4   358.08   both
3      2015-10-01       5     3.99   both
4      2015-10-01       6    47.70   both
...           ...     ...      ...    ...
166973 2015-10-01  505996    21.35   both
166974 2015-10-01  505998    17.69   both
166975 2015-10-01  505999   111.18   both
166976 2015-10-01  506000  1277.92   both
166977 2015-10-01  506002    19.82   both

[166978 rows x 4 col

In [83]:
# TEST

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_item_ltv,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="product_id",
                  timestamp_col_name="timestamp",
                  label_col_name="ltv")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.37 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp      fk    label _merge
0      2016-01-01       0  2448.81   both
1      2016-01-01       1    23.98   both
2      2016-01-01       4   134.28   both
3      2016-01-01       6    23.85   both
4      2016-01-01       7     5.78   both
...           ...     ...      ...    ...
178329 2016-01-01  505998    35.38   both
178330 2016-01-01  505999    74.12   both
178331 2016-01-01  506000  1069.28   both
178332 2016-01-01  506002    19.82   both
178333 2016-01-01  506008    17.00   both

[178334 rows x 4 co

### 4.3 Link Prediction Tasks <a id="amazon-reg-tasks"></a>

#### 4.3.1 user-item-purchase <a id="user-engagement"></a>

Task Description: Predict the list of distinct items each customer will purchase in the next 3 months. 

In [84]:
task_amazon_user_item_purchase = get_task("rel-amazon", "user-item-purchase", download=False)

In [85]:
rtgl_query = """
     PREDICT LIST_DISTINCT(product.*, 0, 91, DAYS)
     FOR EACH customer.*;
"""

In [86]:
# TRAIN

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_item_purchase,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="product_id")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 18.21 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp       fk             label _merge
0       2008-04-10        0         (465326,)   both
1       2010-10-07        0          (93869,)   both
2       2011-04-07        0         (297923,)   both
3       2012-01-05        0         (297644,)   both
4       2012-04-05        0         (413368,)   both
...            ...      ...               ...    ...
5112798 2014-10-02  1850157  (337213, 337946)   both
5112799 2014-10-02  1850158  (337213, 337946)   both
5112800 2015-01-01  1850161         (505350,)   

In [87]:
# VAL

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_item_purchase,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="product_id")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 1.14 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                              label  \
0      2015-10-01        2                       (5251, 25178, 53881, 273172)   
1      2015-10-01        3                                   (384763, 420177)   
2      2015-10-01        5                              (26989, 70766, 81149)   
3      2015-10-01       17                                          (259178,)   
4      2015-10-01       23                                           (30

In [88]:
# TEST

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_item_purchase,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="product_id")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 1.28 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                              label  \
0      2016-01-01        0                                          (125393,)   
1      2016-01-01        2                                        (14, 48618)   
2      2016-01-01        5  (2426, 6311, 81593, 81748, 81751, 81800, 88501...   
3      2016-01-01        8                                   (228518, 410356)   
4      2016-01-01       19                                             

#### 4.3.2 user-item-rate <a id="user-engagement"></a>

Task Description: Predict the list of distinct items each customer will purchase and give a 5 star review in the next 3 months.

In [89]:
task_amazon_user_item_rate = get_task("rel-amazon", "user-item-rate", download=False)

In [90]:
rtgl_query = """
     PREDICT LIST_DISTINCT(review.product_id WHERE review.rating == 5, 0, 91, DAYS)
     FOR EACH customer.*;
"""

In [93]:
# TRAIN

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_item_rate,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="product_id")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 13.65 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp       fk             label _merge
0       2008-04-10        0         (465326,)   both
1       2011-04-07        0         (297923,)   both
2       2012-01-05        0         (297644,)   both
3       2012-04-05        0         (413368,)   both
4       2012-10-04        0         (228080,)   both
...            ...      ...               ...    ...
3667152 2014-10-02  1850157  (337213, 337946)   both
3667153 2014-10-02  1850158  (337213, 337946)   both
3667154 2015-01-01  1850161         (505350,)   

In [94]:
# VAL

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_item_rate,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="product_id")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.87 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                              label  \
0      2015-10-01        2                               (5251, 25178, 53881)   
1      2015-10-01        3                                   (384763, 420177)   
2      2015-10-01        5                              (26989, 70766, 81149)   
3      2015-10-01       17                                          (259178,)   
4      2015-10-01       25                                   (396864, 40

In [95]:
# TEST

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_item_rate,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="product_id")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 0.91 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                label _merge
0      2016-01-01        2                          (14, 48618)   both
1      2016-01-01        5  (2426, 81748, 81751, 81800, 204004)   both
2      2016-01-01        8                     (228518, 410356)   both
3      2016-01-01       19                                 (1,)   both
4      2016-01-01       20                            (237731,)   both
...           ...      ...                        

#### 4.3.3 user-item-review <a id="user-engagement"></a>

Task Description: Predict the list of distinct items each customer will purchase and give a detailed review in the next 3 months.

In [96]:
task_amazon_user_item_review = get_task("rel-amazon", "user-item-review", download=False)

In [97]:
rtgl_query = """
     PREDICT LIST_DISTINCT([
          SELECT
               review.review_text,
               review.product_id,
               review.customer_id,
               review.review_time
          FROM
               review
          WHERE 
               LENGTH(review_text) > 300
            AND
               review_text IS NOT NULL
          ]{new_review}
           {}
           {product_id->product, customer_id->customer}
           {}
           {review_time}.product_id, 0, 91, DAYS)
     FOR EACH customer.*;
"""

In [98]:
# TRAIN

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_item_review,
                  rtgl_query=rtgl_query,
                  split="train",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="product_id")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 12.46 seconds.
------------------- START TRAIN -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
          timestamp       fk             label _merge
0       2008-04-10        0         (465326,)   both
1       2010-10-07        0          (93869,)   both
2       2011-04-07        0         (297923,)   both
3       2012-01-05        0         (297644,)   both
4       2012-04-05        0         (413368,)   both
...            ...      ...               ...    ...
2324172 2009-01-08  1850147         (503320,)   both
2324173 2015-01-01  1850153  (337213, 337946)   both
2324174 2015-01-01  1850154  (337213, 337946)   

In [99]:
# VAL

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_item_review,
                  rtgl_query=rtgl_query,
                  split="val",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="product_id")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1
SQL query executed in 1.51 seconds.
------------------- START VAL -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                              label  \
0      2015-10-01        5                                     (26989, 81149)   
1      2015-10-01       23                                           (30100,)   
2      2015-10-01       25                                    (56023, 396864)   
3      2015-10-01       30                                           (39271,)   
4      2015-10-01       48  (11824, 13491, 41970, 44511, 54705, 68058, 7

In [100]:
# TEST

check_correctness(dataset=dataset_amazon,
                  task=task_amazon_user_item_review,
                  rtgl_query=rtgl_query,
                  split="test",
                  fk_col_name="customer_id",
                  timestamp_col_name="timestamp",
                  label_col_name="product_id")

TIMEDELTA: 91 days 00:00:00
NUM_EVAL_TIMESTAMPS: 1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SQL query executed in 2.70 seconds.
------------------- START TEST -------------------
RelBench fkeys: {'customer_id': 'customer', 'product_id': 'product'}
RelBench pkey: None
RelBench time col: timestamp
RTGL fkeys: {'fk': 'customer', 'label': 'product'}
RTGL pkey: None
RTGL time col: timestamp
Only in RelBench:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
Only in RTGL:
 Empty DataFrame
Columns: [timestamp, fk, label, _merge]
Index: []
In both:
         timestamp       fk                                              label  \
0      2016-01-01        0                                          (125393,)   
1      2016-01-01        5  (6311, 81593, 81748, 81751, 81800, 88501, 2040...   
2      2016-01-01       19                                               (1,)   
3      2016-01-01       20                                          (237731,)   
4      2016-01-01       25                                    (25632, 440766)   
...           ...      ...               